# Gel-transition analysis — cycle-wise LAOS FFT (plus legacy raw-signal SAOS)

**Input:** `observable/*.csv` with `time_s`, `strain`, `shear_rate_1_s`, `stress_Pa`.  
**Optional evaluation only:** corresponding `truth/<sample>_truth.csv`.  
**Output:** per-cycle Fourier moduli, diagnostic figures, event-time comparisons, and CSV files.

### Interpretation and important limitations

- **SAOS:** The **Winter–Chambon (WC) critical-gel criterion** is frequency-independent $\tan\delta=G''/G'$ **across multiple excitation frequencies measured at nearly the same sample age**. A single-frequency SAOS time series **cannot determine a WC gel time**. This notebook reports `not_identifiable` rather than substituting $G'=G''$. The implementation assumes **successive nearly sinusoidal single-frequency cycles** at multiple frequencies, revisited in repeated sweeps, and groups those measurements into short time bins. It does **not** support simultaneous multisine excitation or a continuously sweeping chirp without adaptation. Its time-binned WC event is an *estimate* whose validity depends on sweep speed relative to gelation.
- **LAOS:** Each complete strain cycle is resampled to uniform time points and processed with a **real FFT (`np.fft.rfft`)**. The fundamental complex modulus is $G_1^*=\widehat{\sigma}_1/\widehat{\gamma}_1=G'_1+iG''_1$ (using the positive-frequency convention). The reported event is a **persistent $G'_1>G''_1$ crossover**, which is **not automatically the physical critical gel point**. The third- and fifth-harmonic ratios provide additional nonlinear features.
- **Input mode is an experimental flag**, not reliably deducible from strain amplitude alone. Set `DEFAULT_MODE`, override per sample with `MODE_BY_SAMPLE`, or include a constant `mode`/`oscillation_mode` column (`SAOS`/`LAOS`).
- **Truth never influences detection.** It is read **only after** the event is detected. `injected_clot_state` onset is a synthetic transition-start reference, **not an independently measured critical gel point**. The synthetic truth-derived first-harmonic crossover is compared *only* when its generating-model parameter columns are available.
- **Alert latency:** the retrospectively interpolated LAOS crossing time is distinct from the *confirmation time* after several consecutive cycles. For truly causal online detection, use confirmation time.

### Mathematics

$$
\gamma(t)=\gamma_0\sin(\omega t),\qquad
\sigma(t)=\gamma_0\sum_{n=1,3,5,\dots}\left[G'_n\sin(n\omega t)+G''_n\cos(n\omega t)\right].
$$

$$
G_1^*=\frac{\widehat\sigma_1}{\widehat\gamma_1},\qquad
\tan\delta=\frac{G''_1}{G'_1},\qquad
\left.\frac{\partial\log\tan\delta}{\partial\log\omega}\right|_{t=t_g}\approx0\ \text{(WC candidate)}.
$$

We require a **sign reversal** in that frequency-dependence slope across adjacent, sufficiently populated SAOS time bins before interpolating the WC candidate; a low slope alone is insufficient. Inspect the diagnostics and always validate on independently measured samples.

References: [Winter–Chambon explanation](https://pmc.ncbi.nlm.nih.gov/articles/PMC8955920/) · [FT rheology review](https://www.sciencedirect.com/science/article/pii/S0079670011000281)

### VS Code notebook formatting

This notebook uses Markdown cells (not Python comments) for all explanations and equations.
For VS Code Jupyter, math is written with `$...$` inline and `$$...$$` on separate lines.
For example:

$$
G_1^*(t)=G_1'(t)+iG_1''(t)
$$

Run the code cells with a Python 3 Jupyter kernel; the Markdown cells render automatically.


**Plot style update:** Blue storage modulus, orange loss modulus, green nonlinear harmonics; matching the supplied Section 4 reference. Each figure is exported as PNG and PDF. For APR18 **precomputed** SAOS sweeps, use the separate `SAOS_APR18_Analysis.ipynb`.


## 1. Configuration and imports

Edit `DATA_ROOT` to select `laos_synthetic/` or `APR18_converted/`. Output figures follow the style of the supplied Section 4 notebook.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.ndimage import gaussian_filter1d

# Project root (contains observable/ and optional truth/).
# Set DATA_ROOT to another folder when analysing converted APR18 LAOS CSVs.
DATA_ROOT = Path("laos_synthetic")

# Set to ["clot_A"] for one sample or None to process all observable/*.csv.
SAMPLE_NAMES = None

# The synthetic data in this folder were generated as LAOS examples.
# Choose "SAOS" only for an experiment actually performed in the linear regime.
DEFAULT_MODE = "LAOS"
MODE_BY_SAMPLE = {
    # "clot_A": "LAOS",
    # "saos_sample": "SAOS",
}

# Fourier extraction from complete measured strain cycles.
FFT_POINTS = 512               # uniformly spaced samples PER cycle after interpolation
MIN_RAW_POINTS_PER_CYCLE = 24 # reject undersampled cycles
MAX_STRAIN_HARMONIC_RATIO = 0.20  # QC warning threshold for distorted imposed strain

# LAOS operational crossover detector.
PERSISTENCE_CYCLES = 3

# SAOS Winter-Chambon settings. A time bin should contain a whole repeated
# frequency sweep but be short compared with the sample's gelation timescale.
WC_BIN_SECONDS = 30.0
WC_MIN_DISTINCT_FREQUENCIES = 3
WC_MIN_FREQUENCY_SPAN_RATIO = 1.5
WC_FREQUENCY_CLUSTER_RELATIVE = 0.03  # cluster near-identical excitation f values
WC_LOG_TAN_SPREAD_TOL = 0.35         # max log(tan_delta) spread near WC crossing
WC_MAX_I3_I1_FOR_SAOS = 0.10       # strongly nonlinear cycles excluded from SAOS
WC_MIN_VALID_BINS = 3

# Truth onset when state first rises above its original baseline.
TRUTH_STATE_EPS = 1e-8

OBS_COLS = ["time_s", "strain", "shear_rate_1_s", "stress_Pa"]

# Find data beside this notebook or under data/, regardless of the VS Code cwd.
if not (DATA_ROOT / "observable").is_dir():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (parent / DATA_ROOT,
                          parent / "data" / DATA_ROOT,
                          parent / "Conventional" / DATA_ROOT):
            if (candidate / "observable").is_dir():
                DATA_ROOT = candidate
                break
        if (DATA_ROOT / "observable").is_dir():
            break

OBS_DIR = DATA_ROOT / "observable"
TRUTH_DIR = DATA_ROOT / "truth"
OUT_DIR = Path("results") / "conventional_gel_point" / DATA_ROOT.name

# Plot styling matches Conventional_Rheology_Section4_Analysis.
plt.rcParams.update({
    "figure.dpi": 115, "savefig.dpi": 250,
    "font.size": 10.5, "axes.labelsize": 11, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.23,
    "legend.frameon": False,
})
BLUE, ORANGE, GREEN, GREY = '#2166ac', '#d95f02', '#238b45', '#555555'


def save_figure(fig, stem, folder):
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(folder / f"{stem}.{ext}", bbox_inches="tight", dpi=250)
    plt.show()
    plt.close(fig)
    print("  Saved figures:", folder / f"{stem}.png", "(and PDF)")

print("Observable folder:", OBS_DIR.resolve())
print("Truth folder:     ", TRUTH_DIR.resolve())
print("Results folder:   ", OUT_DIR.resolve())
if not OBS_DIR.is_dir():
    raise FileNotFoundError(
        f"Observable directory not found: {OBS_DIR}\n"
        "Update DATA_ROOT in the configuration cell."
    )
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Data loading and oscillation-mode flag

In [ ]:
def load_observable(path):
    data = pd.read_csv(path)
    missing = [col for col in OBS_COLS if col not in data.columns]
    if missing:
        raise ValueError(f"{path.name}: missing observable columns: {missing}")
    for col in OBS_COLS:
        data[col] = pd.to_numeric(data[col], errors="coerce")
    original_len = len(data)
    data = (data.dropna(subset=OBS_COLS)
                .sort_values("time_s")
                .drop_duplicates("time_s")
                .reset_index(drop=True))
    if len(data) < MIN_RAW_POINTS_PER_CYCLE * 2:
        raise ValueError(f"{path.name}: too few usable observations")
    if len(data) < original_len:
        warnings.warn(f"{path.name}: discarded {original_len-len(data)} invalid/duplicate rows")
    return data


def resolve_mode(data, sample):
    if sample in MODE_BY_SAMPLE:
        mode, source = MODE_BY_SAMPLE[sample], "MODE_BY_SAMPLE"
    else:
        candidate = None
        for col in ["mode", "oscillation_mode", "test_mode"]:
            if col in data.columns:
                unique = data[col].dropna().astype(str).str.strip().str.upper().unique()
                if len(unique) == 1:
                    candidate = unique[0]
                elif len(unique) > 1:
                    raise ValueError(
                        f"{sample}: mixed SAOS/LAOS labels per CSV. Split the file first."
                    )
                break
        mode, source = ((candidate, "CSV flag") if candidate else (DEFAULT_MODE, "DEFAULT_MODE"))
    mode = str(mode).strip().upper()
    if mode not in {"SAOS", "LAOS"}:
        raise ValueError(f"{sample}: mode must be SAOS or LAOS, received {mode!r}")
    return mode, source


def load_truth_if_present(sample):
    file_path = TRUTH_DIR / f"{sample}_truth.csv"
    if not file_path.exists():
        return None
    truth = pd.read_csv(file_path)
    if "time_s" not in truth.columns:
        warnings.warn(f"Truth file for {sample} missing time_s; skipping comparison")
        return None
    truth["time_s"] = pd.to_numeric(truth["time_s"], errors="coerce")
    return (truth.dropna(subset=["time_s"])
                 .sort_values("time_s")
                 .drop_duplicates("time_s")
                 .reset_index(drop=True))


all_files = sorted(OBS_DIR.glob("*.csv"))
selected_files = [p for p in all_files if SAMPLE_NAMES is None or p.stem in SAMPLE_NAMES]
if not selected_files:
    raise FileNotFoundError("No matching CSV files found in observable/. Check SAMPLE_NAMES.")
print("Files to analyse:", ", ".join(p.name for p in selected_files))

## 3. FFT functions: one full strain cycle at a time

We locate consecutive **upward strain zero-crossings** (complete cycles), interpolate strain and stress onto equally spaced points within each cycle, and compute `rfft` on both. The ratio of the *complex first Fourier coefficients* removes the arbitrary phase origin and normalises by the actually measured strain amplitude. Higher-harmonic ratios use **stress** Fourier coefficients. `shear_rate_1_s` is used for sanity checks, not for calculating the first-harmonic modulus.

Assumptions: strain-controlled, approximately sinusoidal excitation within each cycle and a slowly changing material state. Frequency changes must be piecewise constant over each full cycle. Drift, sharp within-cycle frequency changes, noisy zero crossings, and strong strain distortion require inspection of quality-control plots.

In [ ]:
def find_cycle_boundaries(data):
    time = data["time_s"].to_numpy(dtype=float)
    strain = data["strain"].to_numpy(dtype=float)
    # Light smoothing avoids accidental crossings from measurement noise.
    smooth = gaussian_filter1d(strain, sigma=1.0)
    # A genuinely sinusoidal strain should have a nearly constant centre.
    centre = np.median(smooth)
    signed = smooth - centre
    idx = np.flatnonzero((signed[:-1] < 0.0) & (signed[1:] >= 0.0))
    crossing_times = []
    for i in idx:
        frac = -signed[i] / (signed[i+1] - signed[i]) if signed[i+1] != signed[i] else 0.0
        crossing_times.append(time[i] + frac * (time[i+1] - time[i]))
    return np.array(crossing_times, dtype=float)


def extract_cycles_fft(data):
    t = data["time_s"].to_numpy(dtype=float)
    gamma = data["strain"].to_numpy(dtype=float)
    sigma = data["stress_Pa"].to_numpy(dtype=float)
    shear_rate = data["shear_rate_1_s"].to_numpy(dtype=float)
    boundaries = find_cycle_boundaries(data)
    if len(boundaries) < 3:
        raise ValueError("Need multiple complete sinusoidal strain cycles; check time and strain")
    rows = []
    dropped = 0
    for k, (t0, t1) in enumerate(zip(boundaries[:-1], boundaries[1:])):
        lo, hi = np.searchsorted(t, [t0, t1], side="left")
        if hi - lo < MIN_RAW_POINTS_PER_CYCLE or t1 <= t0:
            dropped += 1
            continue
        local_t = t0 + (t1 - t0) * np.arange(FFT_POINTS) / FFT_POINTS
        strain_cycle = np.interp(local_t, t, gamma)
        stress_cycle = np.interp(local_t, t, sigma)
        rate_cycle = np.interp(local_t, t, shear_rate)
        # For exact complete cycles, rectangular-window FFT is appropriate.
        gamma_f = np.fft.rfft(strain_cycle - strain_cycle.mean()) / FFT_POINTS
        sigma_f = np.fft.rfft(stress_cycle - stress_cycle.mean()) / FFT_POINTS
        first_strain = gamma_f[1]
        first_stress = sigma_f[1]
        g0 = 2 * abs(first_strain)
        if g0 < 1e-9 or abs(first_stress) < 1e-12:
            dropped += 1
            continue
        complex_modulus = first_stress / first_strain
        i1 = 2 * abs(first_stress)
        i3 = 2 * abs(sigma_f[3])
        i5 = 2 * abs(sigma_f[5])
        strain_i3_over_i1 = abs(gamma_f[3]) / abs(first_strain)
        f_hz = 1.0 / (t1 - t0)
        omega = 2.0 * np.pi * f_hz
        gp, gpp = complex_modulus.real, complex_modulus.imag
        # Check imposed shear rate against derivative of sinusoidal strain.
        rate_amp_est = np.sqrt(2) * np.std(rate_cycle)
        rate_amp_expected = omega * g0
        rows.append({
            "cycle": k,
            "cycle_start_s": t0,
            "cycle_end_s": t1,
            "time_mid_s": (t0 + t1) / 2,
            "frequency_Hz": f_hz,
            "omega_rad_s": omega,
            "raw_samples": int(hi - lo),
            "strain_amplitude": g0,
            "G1_prime_Pa": gp,
            "G1_double_prime_Pa": gpp,
            "G1_star_Pa": abs(complex_modulus),
            "tan_delta": (gpp / gp if gp > 0 and gpp >= 0 else np.nan),
            "phase_delta_deg": np.degrees(np.angle(complex_modulus)),
            "I1_stress_Pa": i1,
            "I3_stress_Pa": i3,
            "I5_stress_Pa": i5,
            "I3_over_I1": i3 / i1,
            "I5_over_I1": i5 / i1,
            "THD_3_5": np.hypot(i3, i5) / i1,
            "strain_I3_over_I1": strain_i3_over_i1,
            "shear_rate_amplitude_check_ratio": (
                rate_amp_est / rate_amp_expected if rate_amp_expected > 0 else np.nan
            ),
        })
    cycles = pd.DataFrame(rows)
    if cycles.empty:
        raise ValueError("No valid complete cycles; check sample rate and strain excitation")
    print(f"  FFT cycles: {len(cycles):,} retained; {dropped} excluded")
    distorted_fraction = (cycles["strain_I3_over_I1"] > MAX_STRAIN_HARMONIC_RATIO).mean()
    if distorted_fraction > 0.10:
        warnings.warn(
            f"{distorted_fraction:.1%} of cycles have distorted imposed strain (>"
            f"{MAX_STRAIN_HARMONIC_RATIO:.0%} third harmonic). Review excitation."
        )
    return cycles

## 4. Detection functions — independent of the truth file

In [ ]:
def interpolate_zero(x0, y0, x1, y1):
    if not all(np.isfinite([x0, y0, x1, y1])) or y1 == y0:
        return np.nan
    return float(x0 - y0 * (x1 - x0) / (y1 - y0))


def first_persistent_crossover(table, gprime_col, gdouble_col, persistence=3):
    """Return retrospective crossing and causal confirmation times."""
    d = table.sort_values("time_mid_s").reset_index(drop=True)
    diff = (d[gprime_col] - d[gdouble_col]).to_numpy(dtype=float)
    x = d["time_mid_s"].to_numpy(dtype=float)
    confirmation = (d["cycle_end_s"].to_numpy(dtype=float)
                    if "cycle_end_s" in d else x)
    for i in range(1, len(d)):
        if not (np.isfinite(diff[i - 1]) and np.isfinite(diff[i])):
            continue
        if diff[i - 1] >= 0 or diff[i] < 0:
            continue
        j_end = i + persistence
        if j_end > len(d) or not np.all(np.isfinite(diff[i:j_end]) & (diff[i:j_end] > 0)):
            continue
        crossed = interpolate_zero(x[i-1], diff[i-1], x[i], diff[i])
        return {
            "detected_time_s": crossed,
            "confirmation_time_s": float(confirmation[j_end - 1]),
            "status": "detected_operational_G1_crossover",
        }
    return {"detected_time_s": np.nan, "confirmation_time_s": np.nan,
            "status": "no_persistent_G1_crossover"}


def saos_wc_bins(cycles):
    """Approximate multifrequency WC analysis for repeated *single-tone* sweeps."""
    d = cycles.copy()
    d = d.loc[
        (d["G1_prime_Pa"] > 0)
        & (d["G1_double_prime_Pa"] > 0)
        & (d["I3_over_I1"] <= WC_MAX_I3_I1_FOR_SAOS)
        & np.isfinite(d["tan_delta"])
        & (d["frequency_Hz"] > 0)
    ].copy()
    if d.empty:
        return pd.DataFrame(), {"detected_time_s": np.nan,
                                "confirmation_time_s": np.nan,
                                "status": "not_identifiable_no_linear_positive_cycles"}
    # Round frequencies on a logarithmic scale so tiny cycle-period variations
    # do not create spurious 'different frequencies'.
    d["freq_cluster"] = np.rint(
        np.log(d["frequency_Hz"]) / np.log1p(WC_FREQUENCY_CLUSTER_RELATIVE)
    ).astype(int)
    d["age_bin"] = np.floor(
        (d["time_mid_s"] - cycles["cycle_start_s"].min()) / WC_BIN_SECONDS
    ).astype(int)
    grouped = (d.groupby(["age_bin", "freq_cluster"], as_index=False)
        .agg(time_mid_s=("time_mid_s", "median"),
             frequency_Hz=("frequency_Hz", "median"),
             Gp=("G1_prime_Pa", "median"),
             Gpp=("G1_double_prime_Pa", "median"),
             n_cycles=("cycle", "size")))
    grouped["tan_delta"] = grouped["Gpp"] / grouped["Gp"]
    results = []
    for agebin, grp in grouped.groupby("age_bin"):
        if len(grp) < WC_MIN_DISTINCT_FREQUENCIES:
            continue
        span = grp["frequency_Hz"].max() / grp["frequency_Hz"].min()
        if span < WC_MIN_FREQUENCY_SPAN_RATIO:
            continue
        log_f = np.log(grp["frequency_Hz"].to_numpy())
        log_tan = np.log(grp["tan_delta"].to_numpy())
        slope = float(np.polyfit(log_f, log_tan, 1)[0])
        log_spread = float(np.max(log_tan) - np.min(log_tan))
        gp_exp = float(np.polyfit(log_f, np.log(grp["Gp"]), 1)[0])
        gpp_exp = float(np.polyfit(log_f, np.log(grp["Gpp"]), 1)[0])
        results.append({
            "age_bin": int(agebin),
            "time_mid_s": float(grp["time_mid_s"].median()),
            "frequency_count": len(grp),
            "min_frequency_Hz": float(grp["frequency_Hz"].min()),
            "max_frequency_Hz": float(grp["frequency_Hz"].max()),
            "frequency_span_ratio": span,
            "slope_log_tandelta_vs_log_f": slope,
            "log_tandelta_spread": log_spread,
            "n_Gprime": gp_exp,
            "n_Gdoubleprime": gpp_exp,
            "tan_delta_median": float(grp["tan_delta"].median()),
        })
    bins = pd.DataFrame(results)
    if len(bins) < WC_MIN_VALID_BINS:
        return bins, {"detected_time_s": np.nan,
                      "confirmation_time_s": np.nan,
                      "status": "not_identifiable_insufficient_multifrequency_time_bins"}
    bins = bins.sort_values("time_mid_s").reset_index(drop=True)
    # Frequency-independent tan(delta) must be temporally bracketed by slopes
    # of opposite signs; reject an isolated near-flat plateau/noisy minimum.
    best = None
    for i in range(1, len(bins)):
        left, right = bins.iloc[i-1], bins.iloc[i]
        s0, s1 = left["slope_log_tandelta_vs_log_f"], right["slope_log_tandelta_vs_log_f"]
        if s0 * s1 > 0 or s0 == s1:
            continue
        if right["time_mid_s"] - left["time_mid_s"] > 2.0 * WC_BIN_SECONDS:
            continue
        if min(left["log_tandelta_spread"], right["log_tandelta_spread"]) > WC_LOG_TAN_SPREAD_TOL:
            continue
        tcandidate = interpolate_zero(left["time_mid_s"], s0,
                                      right["time_mid_s"], s1)
        if not np.isfinite(tcandidate):
            continue
        # AWC note: timing resolution is limited by whole repeated sweep bin duration.
        best = {
            "detected_time_s": tcandidate,
            "confirmation_time_s": float(right["time_mid_s"] + WC_BIN_SECONDS/2),
            "status": "detected_WC_candidate_multifrequency",
        }
        break
    if best is None:
        best = {"detected_time_s": np.nan,
                "confirmation_time_s": np.nan,
                "status": "no_bracketed_WC_frequency_independence"}
    return bins, best

## 5. Evaluation only (load truth **after** detection)

The `injected_clot_state` reference reports the first synthetic state change. It does **not** establish the true critical gel point. For the specific older LAOS generator, we may also compare our FFT crossover against a *synthetic-model-derived first-harmonic crossover*. Do **not** use that equation for a different physical/synthetic model.

In [ ]:
def truth_onset(truth):
    if truth is None or "injected_clot_state" not in truth:
        return np.nan
    state = pd.to_numeric(truth["injected_clot_state"], errors="coerce").to_numpy()
    if len(state) == 0 or not np.isfinite(state).any():
        return np.nan
    baseline = state[np.flatnonzero(np.isfinite(state))[0]]
    ids = np.flatnonzero(np.isfinite(state) & (state > baseline + TRUTH_STATE_EPS))
    return float(truth["time_s"].iloc[ids[0]]) if len(ids) else np.nan


def annotated_gel_point(truth):
    if truth is None:
        return np.nan
    for col in ["true_gel_point_s", "critical_gel_time_s", "gel_point_time_s",
                "true_gel_time_s", "gel_time_s"]:
        if col in truth.columns:
            values = pd.to_numeric(truth[col], errors="coerce").dropna()
            if len(values):
                return float(values.iloc[0])
    return np.nan


def synthetic_model_truth_per_cycle(truth, cycles):
    required = ["ideal_Gprime_scale_Pa", "ideal_Gdoubleprime_scale_Pa",
                "ideal_third_order_a", "ideal_fifth_order_b"]
    if truth is None or any(c not in truth.columns for c in required):
        return pd.DataFrame()
    tref = truth["time_s"].to_numpy(dtype=float)
    for col in required:
        # Guard against nonnumeric or missing truth parameters.
        if not pd.to_numeric(truth[col], errors="coerce").notna().all():
            return pd.DataFrame()
    # This analytic transformation is SPECIFIC to x - a*x^3 + b*x^5.
    gp_scale = truth[required[0]].to_numpy(dtype=float)
    gpp_scale = truth[required[1]].to_numpy(dtype=float)
    a = truth[required[2]].to_numpy(dtype=float)
    b = truth[required[3]].to_numpy(dtype=float)
    g1p = gp_scale * (1.0 - 0.75*a + 0.625*b)
    g1pp = gpp_scale
    rows = []
    for rec in cycles.itertuples(index=False):
        lo, hi = np.searchsorted(tref, [rec.cycle_start_s, rec.cycle_end_s], side="left")
        if hi > lo:
            p, q = float(np.mean(g1p[lo:hi])), float(np.mean(g1pp[lo:hi]))
        else:
            p = float(np.interp(rec.time_mid_s, tref, g1p))
            q = float(np.interp(rec.time_mid_s, tref, g1pp))
        rows.append({
            "cycle": rec.cycle,
            "time_mid_s": rec.time_mid_s,
            "cycle_end_s": rec.cycle_end_s,
            "G1_prime_truth_Pa": p,
            "G1_double_prime_truth_Pa": q,
        })
    return pd.DataFrame(rows)


def evaluate_with_truth(truth, cycles, mode):
    onset = truth_onset(truth)
    annotated = annotated_gel_point(truth)
    truth_fft = (synthetic_model_truth_per_cycle(truth, cycles)
                 if mode == "LAOS" else pd.DataFrame())
    reference_crossing = np.nan
    if not truth_fft.empty:
        ref = first_persistent_crossover(
            truth_fft, "G1_prime_truth_Pa", "G1_double_prime_truth_Pa", persistence=1
        )
        reference_crossing = ref["detected_time_s"]
    return {"truth_onset_s": onset,
            "truth_annotated_critical_gel_s": annotated,
            "truth_ideal_G1_crossover_s": reference_crossing}, truth_fft

## 6. Plots — shared Section 4 styling

The LAOS FFT plots use consistent blue/orange/green colors, light grids, frameless legends and vector PDF + PNG exports. Operational crossovers should not be described as confirmed critical gel points.

In [ ]:
def mark_events(ax, detection, truth_summary):
    """Consistent event markers (ground truth shown only when available)."""
    markers = [
        (detection.get("detected_time_s"), "Observable detection", '#b2182b', '-'),
        (truth_summary.get("truth_onset_s"), "Synthetic onset", GREEN, '--'),
        (truth_summary.get("truth_annotated_critical_gel_s"), "Annotated critical gel", '#762a83', '--'),
        (truth_summary.get("truth_ideal_G1_crossover_s"), "Ideal synthetic crossover", ORANGE, ':'),
    ]
    for value, name, color, style in markers:
        if value is not None and np.isfinite(value):
            ax.axvline(value, ls=style, color=color, lw=1.5, alpha=.85, label=name)


def plot_laos(sample, cycles, detection, truth_summary, truth, ideal, output_dir):
    t = cycles["time_mid_s"]
    fig, axes = plt.subplots(4, 1, figsize=(10.2, 11.8), sharex=True,
                            gridspec_kw={"height_ratios": [1.4, 0.85, 0.95, 0.9]})
    axes[0].plot(t, cycles["G1_prime_Pa"], color=BLUE, lw=1.9,
                 label=r"Measured $G'_1$")
    axes[0].plot(t, cycles["G1_double_prime_Pa"], color=ORANGE, lw=1.9,
                 label=r"Measured $G''_1$")
    if not ideal.empty:
        axes[0].plot(ideal["time_mid_s"], ideal["G1_prime_truth_Pa"], '--',
                     color=BLUE, lw=1.25, alpha=.6, label=r"Ideal synthetic $G'_1$")
        axes[0].plot(ideal["time_mid_s"], ideal["G1_double_prime_truth_Pa"], '--',
                     color=ORANGE, lw=1.25, alpha=.6, label=r"Ideal synthetic $G''_1$")
    axes[0].set(ylabel="First-harmonic modulus (Pa)",
                title=f"{sample} — cycle-wise LAOS FFT and operational crossover")
    axes[1].plot(t, cycles["G1_prime_Pa"] - cycles["G1_double_prime_Pa"],
                 color=BLUE, lw=1.6)
    axes[1].axhline(0, color=GREY, ls='--', lw=1.2)
    axes[1].set_ylabel(r"$G'_1 - G''_1$ (Pa)")
    axes[2].plot(t, cycles["I3_over_I1"], color=GREEN, lw=1.75,
                 label=r"$I_3/I_1$")
    axes[2].plot(t, cycles["I5_over_I1"], color=ORANGE, lw=1.65, ls='--',
                 label=r"$I_5/I_1$")
    axes[2].set_ylabel("Stress harmonic ratio")
    if truth is not None and "injected_clot_state" in truth:
        axes[3].plot(truth["time_s"], truth["injected_clot_state"], color=GREEN,
                     lw=1.8, label="Synthetic injected state (evaluation only)")
        axes[3].set_ylabel("Injected state")
    else:
        axes[3].plot(t, cycles["frequency_Hz"], color=GREY, lw=1.6,
                     label="Excitation frequency (Hz)")
        axes[3].set_ylabel("Frequency (Hz)")
    for ax in axes:
        mark_events(ax, detection, truth_summary)
    axes[0].legend(fontsize=8.2, ncol=2)
    axes[2].legend(loc="best")
    axes[3].legend(loc="best", fontsize=8.5)
    axes[-1].set_xlabel("Time (s)")
    save_figure(fig, f"{sample}_LAOS_full", output_dir)

    centre = detection.get("detected_time_s", np.nan)
    if not np.isfinite(centre):
        centre = truth_summary.get("truth_onset_s", np.nan)
    if np.isfinite(centre):
        halfwidth = 30.0
        fig, ax = plt.subplots(figsize=(8.8, 5.2))
        mask = t.between(centre - halfwidth, centre + halfwidth)
        ax.plot(t[mask], cycles.loc[mask, "G1_prime_Pa"], 'o-', color=BLUE,
                ms=3.2, lw=1.8, label=r"Measured $G'_1$")
        ax.plot(t[mask], cycles.loc[mask, "G1_double_prime_Pa"], 's-',
                color=ORANGE, ms=3.2, lw=1.8, label=r"Measured $G''_1$")
        if not ideal.empty:
            m = ideal["time_mid_s"].between(centre - halfwidth, centre + halfwidth)
            ax.plot(ideal.loc[m, "time_mid_s"], ideal.loc[m, "G1_prime_truth_Pa"], '--',
                    color=BLUE, lw=1.4, alpha=.65, label=r"Ideal $G'_1$")
            ax.plot(ideal.loc[m, "time_mid_s"], ideal.loc[m, "G1_double_prime_truth_Pa"], '--',
                    color=ORANGE, lw=1.4, alpha=.65, label=r"Ideal $G''_1$")
        mark_events(ax, detection, truth_summary)
        ax.set_xlim(centre-halfwidth, centre+halfwidth)
        ax.set(xlabel="Time (s)", ylabel="First-harmonic modulus (Pa)",
               title=f"{sample} — detail near detected transition")
        ax.legend(fontsize=8, ncol=2)
        save_figure(fig, f"{sample}_LAOS_zoom", output_dir)


def plot_saos(sample, cycles, bins, detection, truth_summary, output_dir):
    """Retained support for original *raw-signal* SAOS mode (not APR18-format SAOS)."""
    fig, axes = plt.subplots(3, 1, figsize=(9.0, 9.5), sharex=True)
    valid = cycles.loc[(cycles["G1_prime_Pa"] > 0) & (cycles["G1_double_prime_Pa"] > 0)].copy()
    if not valid.empty:
        valid["frequency_group"] = np.rint(
            np.log(valid["frequency_Hz"]) / np.log1p(WC_FREQUENCY_CLUSTER_RELATIVE)
        ).astype(int)
        for _, sub in valid.groupby("frequency_group"):
            label = f"{sub['frequency_Hz'].median():.3g} Hz"
            axes[0].scatter(sub["time_mid_s"], sub["tan_delta"], s=17,
                            alpha=.65, label=label)
        if valid["frequency_group"].nunique() <= 12:
            axes[0].legend(fontsize=8, ncol=3)
        axes[0].set_yscale("log")
    axes[0].set(ylabel=r"Loss tangent $\tan\delta$",
                title=f"{sample} — SAOS Winter–Chambon diagnostics")
    if not bins.empty:
        axes[1].plot(bins["time_mid_s"], bins["slope_log_tandelta_vs_log_f"],
                     'o-', color=BLUE, ms=3, lw=1.75)
        axes[2].plot(bins["time_mid_s"], bins["log_tandelta_spread"],
                     'D-', color=GREEN, ms=3, lw=1.75,
                     label=r"Log $\tan\delta$ spread")
    axes[1].axhline(0, color=GREY, ls='--', lw=1.1)
    axes[1].set_ylabel(r"$d\log\tan\delta/d\log f$")
    axes[2].axhline(WC_LOG_TAN_SPREAD_TOL, color=GREY, ls=':',
                    label="Heuristic tolerance")
    axes[2].set_ylabel(r"Log $\tan\delta$ spread")
    for ax in axes:
        mark_events(ax, detection, truth_summary)
    axes[2].legend(fontsize=8)
    axes[-1].set_xlabel("Time (s)")
    save_figure(fig, f"{sample}_SAOS_WC", output_dir)

## 7. Run detection, then independently evaluate against truth

In [ ]:
summary_rows = []
all_results = {}

for input_path in selected_files:
    sample = input_path.stem
    print("\n" + "="*70 + f"\n{sample}\n" + "="*70)
    try:
        # OBSERVABLE-ONLY DETECTION: no truth is available to these functions.
        observed = load_observable(input_path)
        mode, mode_source = resolve_mode(observed, sample)
        print(f"  Mode: {mode} (from {mode_source})")
        print(f"  Rows: {len(observed):,}")
        cycles = extract_cycles_fft(observed)
        wc_bins = pd.DataFrame()
        if mode == "LAOS":
            detection = first_persistent_crossover(
                cycles, "G1_prime_Pa", "G1_double_prime_Pa",
                persistence=PERSISTENCE_CYCLES
            )
            method = "FFT G'1/G''1 persistent crossover (NOT critical gel criterion)"
        else:
            wc_bins, detection = saos_wc_bins(cycles)
            method = "Winter-Chambon (multi-frequency SAOS only)"

        # TRUTH EVALUATION: intentionally occurs AFTER event detection.
        truth = load_truth_if_present(sample)
        truth_summary, ideal = evaluate_with_truth(truth, cycles, mode)
        detected_t = detection["detected_time_s"]
        onset_t = truth_summary["truth_onset_s"]
        critical_t = truth_summary["truth_annotated_critical_gel_s"]
        ideal_t = truth_summary["truth_ideal_G1_crossover_s"]
        row = {
            "sample": sample,
            "mode": mode,
            "method": method,
            "status": detection["status"],
            "observable_detection_s": detected_t,
            "observable_confirmation_s": detection["confirmation_time_s"],
            "truth_injected_clot_state_onset_s": onset_t,
            "truth_annotated_critical_gel_s": critical_t,
            "truth_ideal_G1_crossover_s": ideal_t,
            "detection_minus_onset_s": (detected_t - onset_t
                                         if np.isfinite(detected_t) and np.isfinite(onset_t) else np.nan),
            "detection_minus_annotated_gel_s": (detected_t - critical_t
                                                if np.isfinite(detected_t) and np.isfinite(critical_t) else np.nan),
            "detection_minus_ideal_G1_crossover_s": (detected_t - ideal_t
                                                     if np.isfinite(detected_t) and np.isfinite(ideal_t) else np.nan),
            "usable_fft_cycles": len(cycles),
            "median_frequency_Hz": float(cycles["frequency_Hz"].median()),
            "median_strain_amplitude": float(cycles["strain_amplitude"].median()),
            "median_I3_over_I1": float(cycles["I3_over_I1"].median()),
            "valid_WC_time_bins": len(wc_bins),
        }
        summary_rows.append(row)
        all_results[sample] = {"cycles": cycles, "wc_bins": wc_bins,
                               "detection": detection, "truth_summary": truth_summary,
                               "ideal": ideal, "mode": mode}
        sample_dir = OUT_DIR / sample
        sample_dir.mkdir(parents=True, exist_ok=True)
        cycles.to_csv(sample_dir / f"{sample}_fft_cycle_moduli.csv", index=False)
        if not wc_bins.empty:
            wc_bins.to_csv(sample_dir / f"{sample}_WC_time_bins.csv", index=False)
        if not ideal.empty:
            ideal.to_csv(sample_dir / f"{sample}_synthetic_truth_first_harmonics.csv", index=False)
        pd.DataFrame([row]).to_csv(sample_dir / f"{sample}_comparison.csv", index=False)
        if mode == "LAOS":
            plot_laos(sample, cycles, detection, truth_summary, truth, ideal, sample_dir)
        else:
            plot_saos(sample, cycles, wc_bins, detection, truth_summary, sample_dir)
        print("  Detection:", detection["status"])
        print("  Estimated time (s):", detected_t)
        print("  Truth injected onset (s):", onset_t)
        print("  Files saved to:", sample_dir)
    except Exception as exc:
        # Preserve failures in the batch summary instead of silently skipping.
        print(f"  ERROR in {sample}: {type(exc).__name__}: {exc}")
        summary_rows.append({"sample": sample, "mode": None, "method": None,
                             "status": f"ERROR: {type(exc).__name__}: {exc}"})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "comparison_summary_all_samples.csv", index=False)
print("\nCOMPARISON TABLE (time units: seconds)")
display(summary_df)
print("\nSaved:", OUT_DIR / "comparison_summary_all_samples.csv")

## 8. How to interpret the saved results

**LAOS mode:**

- `observable_detection_s` is the estimated **first-harmonic storage/loss crossover**, not a validated physical gel time.
- `observable_confirmation_s` is when the required successive elastic-dominant cycles have been observed; the confirmed result cannot be available before this time.
- `I3_over_I1` and `I5_over_I1` indicate nonlinear stress harmonics. The first-harmonic modulus alone discards this information.
- `truth_ideal_G1_crossover_s` comes from the **known synthetic generator** and is **only used in evaluation**. Differences should be compared against this truth *separately* from differences against the injected clot-state onset.

**SAOS mode:**

- A valid Winter–Chambon event requires multi-frequency observations close together in material age. If the file contains only ~1 Hz, the notebook reports **`not_identifiable_insufficient_multifrequency_time_bins`** (not a fabricated WC gel time).
- Review `*_SAOS_WC.png`: the estimated gel time should occur where the frequency-dependence slope changes sign near zero and the loss-tangent curves converge. Confirm linear strain amplitude, frequency coverage, and sweep speed experimentally.
- `WC_BIN_SECONDS` is a **user-controlled modelling assumption**: shorten it if gelation is fast, but it must still contain a complete repeated frequency sweep.
- The `injected_clot_state` onset is **not** a Winter–Chambon ground-truth gel label; `truth_annotated_critical_gel_s` is used only if the truth CSV explicitly provides such an annotation.

**Requirements:** `numpy`, `pandas`, `scipy`, `matplotlib`, and `ipykernel`. In VS Code, select a Python kernel and run the notebook top-to-bottom. If imports fail, use `%pip install numpy pandas scipy matplotlib ipykernel` in a separate cell.

**Note on experimental design:** Fourier decomposition of a single-frequency time series does not create the independent excitation frequencies needed by Winter–Chambon. Higher stress harmonics in LAOS **must not** be treated as separate SAOS frequencies for WC analysis.